# 🔬 KLA Hackathon — NAFNet-SR Image Restoration
**AI-Based Restoration of Degraded Semiconductor Inspection Images**

### Prerequisites (already done ✅):
- `train/` and `Test_NoisyLR/` folders are in `My Drive/kla_data/`
- GPU runtime is set to T4

Click `Runtime → Run all`

In [ ]:
# CELL 1: Check GPU
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# CELL 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

In [ ]:
# CELL 3: Clone repo & install dependencies
import os
REPO_URL = 'https://github.com/norriy0u/kla-image-restoration.git'
REPO_DIR = '/content/kla_restoration'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
%cd {REPO_DIR}
!pip install -r requirements.txt -q
print('✓ Setup complete!')

In [ ]:
# CELL 4: Detect dataset paths and copy to /content for fast I/O
import os, glob

DRIVE_KLA  = '/content/drive/MyDrive/kla_data'
LOCAL_DATA = '/content/kla_data'
os.makedirs(LOCAL_DATA, exist_ok=True)

# ── Step 1: Show exact Drive structure ─────────────────────────
print('=== Google Drive kla_data structure ===')
for root, dirs, files in os.walk(DRIVE_KLA):
    level = root.replace(DRIVE_KLA, '').count(os.sep)
    if level > 4: continue
    n_npy = len([f for f in files if f.endswith('.npy')])
    indent = '  ' * level
    tag = f'({n_npy} .npy)' if n_npy > 0 else ''
    print(f'{indent}{os.path.basename(root)}/  {tag}')

# ── Step 2: Find all directories with .npy files ───────────────
def find_npy_dirs(base, min_files=10):
    results = []
    for root, _, files in os.walk(base):
        n = len([f for f in files if f.endswith('.npy')])
        if n >= min_files:
            results.append((n, root))
    return sorted(results, reverse=True)

drive_npy = find_npy_dirs(DRIVE_KLA)
print('\n=== Directories with .npy files ===')
for n, p in drive_npy:
    print(f'  {n:5d}  {p}')

# ── Step 3: Auto-detect GT / LR / Test directories ────────────
def pick(dirs, *must_have, exclude=()):
    for _, p in dirs:
        name = p.lower()
        if all(k.lower() in name for k in must_have):
            if not any(e.lower() in name for e in exclude):
                return p
    return None

DRIVE_GT   = pick(drive_npy, 'gt',    exclude=('noisy', 'test'))
DRIVE_LR   = pick(drive_npy, 'noisy', exclude=('test',))
DRIVE_TEST = pick(drive_npy, 'test')

# Handle case: Test_NoisyLR has .npy directly (no NoisyLR subfolder)
if not DRIVE_TEST:
    candidate = os.path.join(DRIVE_KLA, 'Test_NoisyLR')
    if os.path.isdir(candidate):
        DRIVE_TEST = candidate

print(f'\nDRIVE_GT   = {DRIVE_GT}')
print(f'DRIVE_LR   = {DRIVE_LR}')
print(f'DRIVE_TEST = {DRIVE_TEST}')

assert DRIVE_GT,   'Could not find GT directory — see tree above'
assert DRIVE_LR,   'Could not find NoisyLR directory — see tree above'
assert DRIVE_TEST, 'Could not find Test directory — see tree above'

# ── Step 4: Copy Drive → /content (10x faster for training) ───
GT_DIR   = os.path.join(LOCAL_DATA, 'GT')
LR_DIR   = os.path.join(LOCAL_DATA, 'NoisyLR')
TEST_DIR = os.path.join(LOCAL_DATA, 'Test_NoisyLR')

for src, dst, label in [(DRIVE_GT, GT_DIR, 'GT'), (DRIVE_LR, LR_DIR, 'NoisyLR'), (DRIVE_TEST, TEST_DIR, 'Test')]:
    if not os.path.exists(dst):
        print(f'Copying {label} from Drive → /content ...')
        !cp -r "{src}" "{dst}"
        print(f'  ✓ Done')
    else:
        n = len(glob.glob(os.path.join(dst, '*.npy')))
        print(f'  {label} already in /content ({n} files)')

# ── Step 5: Verify ─────────────────────────────────────────────
gt_n   = len(glob.glob(os.path.join(GT_DIR,   '*.npy')))
lr_n   = len(glob.glob(os.path.join(LR_DIR,   '*.npy')))
test_n = len(glob.glob(os.path.join(TEST_DIR, '*.npy')))
print(f'\nGT: {gt_n} | LR: {lr_n} | Test: {test_n}')
assert gt_n > 0 and lr_n > 0, f'No .npy files! GT_DIR={GT_DIR}, LR_DIR={LR_DIR}'
print('✓ Dataset ready!')

In [ ]:
# CELL 5: Quick data inspection
import numpy as np, matplotlib.pyplot as plt, glob, os

gt_files = sorted(glob.glob(os.path.join(GT_DIR, '*.npy')))
lr_files = sorted(glob.glob(os.path.join(LR_DIR, '*.npy')))
print(f'GT: {len(gt_files)} | LR: {len(lr_files)}')

n = len(gt_files)
indices = [0, n//3, 2*n//3, n-1]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col, idx in enumerate(indices):
    gt = np.load(gt_files[idx])
    lr = np.load(lr_files[idx])
    axes[0][col].imshow(gt, cmap='gray', vmin=0, vmax=1)
    axes[0][col].set_title(f'GT #{idx} {gt.shape}\n[{gt.min():.2f},{gt.max():.2f}]', fontsize=9)
    axes[0][col].axis('off')
    axes[1][col].imshow(np.clip(lr,0,1), cmap='gray', vmin=0, vmax=1)
    axes[1][col].set_title(f'NoisyLR #{idx} {lr.shape}\nmax={lr.max():.2f}', fontsize=9)
    axes[1][col].axis('off')

plt.suptitle('Training Pairs — GT (256×256) vs NoisyLR (128×128)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/sample_pairs.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: /content/sample_pairs.png')

In [ ]:
# CELL 6: Train! (~2h on T4, ~45min on A100)
BATCH_SIZE = 8   # use 16 for A100
EPOCHS = 200

!python train.py \
    --gt_dir {GT_DIR} \
    --lr_dir {LR_DIR} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --num_workers 2 \
    --val_fraction 0.1 \
    --patch_size_gt 256 \
    --model_variant base \
    --weights_dir ./weights \
    --log_dir ./logs

In [ ]:
# CELL 7: TensorBoard
%load_ext tensorboard
%tensorboard --logdir ./logs

In [ ]:
# CELL 8: Evaluate on validation set
import os, sys, json
sys.path.insert(0, '.')
os.makedirs('/content/val_outputs', exist_ok=True)
!python evaluate.py \
    --input_dir {LR_DIR} \
    --output_dir /content/val_outputs \
    --gt_dir {GT_DIR} \
    --weights ./weights/best_model.pt \
    --batch_size 8
with open('/content/val_outputs/metrics.json') as f:
    m = json.load(f)
print('\n=== VALIDATION METRICS ===')
for k, v in m.items():
    print(f'  {k}: {v}')

In [ ]:
# CELL 9: Run inference on official test set
import os, json
os.makedirs('/content/test_outputs', exist_ok=True)
!python evaluate.py \
    --input_dir {TEST_DIR} \
    --output_dir /content/test_outputs \
    --weights ./weights/best_model.pt \
    --batch_size 8
with open('/content/test_outputs/metrics.json') as f:
    print(json.dumps(json.load(f), indent=2))

In [ ]:
# CELL 10: Before → After → GT comparison (PPT Slide 6)
import numpy as np, matplotlib.pyplot as plt, glob, os

gt_files = sorted(glob.glob(os.path.join(GT_DIR, '*.npy')))
lr_files = sorted(glob.glob(os.path.join(LR_DIR, '*.npy')))
pool = lr_files[:min(300, len(lr_files))]
sample_lr = [f for _, f in sorted([(np.load(f).max(), f) for f in pool], reverse=True)[:4]]

fig, axes = plt.subplots(4, 3, figsize=(15, 20))
titles = ['NoisyLR Input (128×128)', 'NAFNet-SR Output (256×256)', 'Ground Truth (256×256)']
colors = ['#e74c3c', '#2ecc71', '#3498db']

for row, lr_path in enumerate(sample_lr):
    stem    = os.path.splitext(os.path.basename(lr_path))[0]
    gt_path = os.path.join(GT_DIR, f'{stem}.npy')
    out_npy = f'/content/val_outputs/{stem}.npy'
    lr_arr  = np.load(lr_path)
    gt_arr  = np.load(gt_path) if os.path.exists(gt_path) else np.zeros((256,256))
    out_arr = np.load(out_npy)  if os.path.exists(out_npy)  else np.zeros((256,256))
    for col, (arr, t, c) in enumerate(zip([lr_arr, out_arr, gt_arr], titles, colors)):
        axes[row][col].imshow(np.clip(arr,0,1), cmap='gray', vmin=0, vmax=1)
        axes[row][col].set_title(f'{t}\n[{arr.min():.2f},{arr.max():.2f}]',
                                  fontsize=10, color=c, fontweight='bold')
        axes[row][col].axis('off')
    axes[row][0].set_ylabel(f'#{stem}', fontsize=9, rotation=90, labelpad=12)

plt.suptitle('NAFNet-SR: Degraded → Restored → Ground Truth', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/before_after_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved /content/before_after_comparison.png  ← use in PPT Slide 6!')

In [ ]:
# CELL 11: Save everything to Drive
import shutil, os
DRIVE_OUT = '/content/drive/MyDrive/kla_submission'
os.makedirs(DRIVE_OUT, exist_ok=True)
shutil.copy('./weights/best_model.pt',              f'{DRIVE_OUT}/best_model.pt')
shutil.copytree('/content/test_outputs',            f'{DRIVE_OUT}/test_outputs',            dirs_exist_ok=True)
shutil.copy('/content/before_after_comparison.png', f'{DRIVE_OUT}/before_after_comparison.png')
shutil.copy('/content/sample_pairs.png',            f'{DRIVE_OUT}/sample_pairs.png')
print(f'✓ All saved to: {DRIVE_OUT}')

In [ ]:
# CELL 12: [OPTIONAL] TTA for best quality
import os, json
os.makedirs('/content/test_outputs_tta', exist_ok=True)
!python evaluate.py \
    --input_dir {TEST_DIR} \
    --output_dir /content/test_outputs_tta \
    --weights ./weights/best_model.pt \
    --tta --batch_size 1
with open('/content/test_outputs_tta/metrics.json') as f:
    print(json.dumps(json.load(f), indent=2))